In [1]:
import pandas as pd
import matplotlib.pyplot as plt

lmp = pd.read_parquet("../data/processed/lmp_2025.parquet")

In [2]:
# Isolating the BGE and Western Hub day ahead data
bge_da_iso = lmp[(lmp["market"] == "DA") & (lmp["node"] == "BGE")]
wh_da_iso = lmp[(lmp["market"] == "DA") & (lmp["node"] == "WESTERN HUB")]

In [3]:
# Targeting the BGE LMP, Congestion, and Loss columns
bge_da_iso = bge_da_iso[["lmp","congestion","loss"]]

In [4]:
# Targeting the WH LMP, Congestion, and Loss columns
wh_da_iso = wh_da_iso[["lmp","congestion","loss"]]

In [5]:
bge_da_iso.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 8760 entries, 2025-01-01 05:00:00+00:00 to 2026-01-01 04:00:00+00:00
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   lmp         8760 non-null   float64
 1   congestion  8760 non-null   float64
 2   loss        8760 non-null   float64
dtypes: float64(3)
memory usage: 273.8 KB


In [6]:
wh_da_iso.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 8760 entries, 2025-01-01 05:00:00+00:00 to 2026-01-01 04:00:00+00:00
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   lmp         8760 non-null   float64
 1   congestion  8760 non-null   float64
 2   loss        8760 non-null   float64
dtypes: float64(3)
memory usage: 273.8 KB


In [7]:
# Creating a new table that contains the hourly difference between BGE and WH prices
price_diff = (bge_da_iso - wh_da_iso)

In [8]:
price_diff.describe()

,lmp,congestion,loss
count,8760.000000,8760.000000,8760.000000
mean,7.126420,5.721294,1.405126
std,15.314733,14.194111,1.731991
min,-78.183835,-79.612198,-0.601255
25%,0.804308,0.000000,0.538582
50%,3.247057,2.282188,0.960789
75%,9.647221,8.130707,1.652793
max,288.912323,268.712153,30.967041


In [9]:
# Creating an assert to ensure that Congestion + Loss is always equal to the difference in prices between BGE and WH
gap = (price_diff["lmp"]) - ((price_diff["congestion"] + price_diff["loss"]))
assert gap.abs().max() <= 0.01, "Congestion difference and Loss difference does not equal LMP difference (error is greater than $0.01)"

In [10]:
# Converting the index to Eastern time
price_diff.index = price_diff.index.tz_convert("America/New_York")

In [11]:
# Creating a table that highlights the average price difference in each month
monthly_price_diff = price_diff.resample("MS").mean()

In [12]:
monthly_price_diff.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 12 entries, 2025-01-01 00:00:00-05:00 to 2025-12-01 00:00:00-05:00
Freq: MS
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   lmp         12 non-null     float64
 1   congestion  12 non-null     float64
 2   loss        12 non-null     float64
dtypes: float64(3)
memory usage: 384.0 bytes


In [13]:
monthly_price_diff.head()

,lmp,congestion,loss
2025-01-01 00:00:00-05:00,8.585105,6.381297,2.203808
2025-02-01 00:00:00-05:00,1.302011,0.089720,1.212291
2025-03-01 00:00:00-05:00,5.953184,5.294125,0.659059
2025-04-01 00:00:00-04:00,6.616889,5.744317,0.872572
2025-05-01 00:00:00-04:00,5.416634,4.146613,1.270021


In [14]:
# Calculating the mean annual price difference 
price_diff.mean()

lmp           7.126420
congestion    5.721294
loss          1.405126
dtype: float64

In [15]:
# Calculating what percentage of the price difference comes from congestion 
congestion_diff_percentage = (price_diff["congestion"].mean() / price_diff["lmp"].mean())
round(congestion_diff_percentage * 100, 1)

np.float64(80.3)

# Noting that congestion costs account for ~80% of the difference between the BGE and WH LMP  

In [16]:
# Calculating what percentage of the price difference comes from loss 
loss_diff_percentage = (price_diff["loss"].mean() / price_diff["lmp"].mean())
loss_diff_percentage
round(loss_diff_percentage * 100, 1)

np.float64(19.7)

# Noting that loss costs account for ~20% of the difference between the BGE and WH LMP 